# Simple RAG Demo

This Colab notebook builds a complete Retrieval-Augmented Generation system:

`GitHub issues -> chunks -> embeddings -> FAISS -> retriever -> prompt -> Zephyr -> answer`

It follows the Hugging Face Open-Source AI Cookbook recipe and adds inspection cells so every stage is visible.

## 1. Check the Colab GPU

Use **Runtime > Change runtime type > T4 GPU** before continuing. Zephyr-7B is loaded in 4-bit precision.

In [ ]:
!nvidia-smi

## 2. Install dependencies

`bitsandbytes` provides 4-bit quantization, `sentence-transformers` provides embeddings, FAISS stores and searches vectors, and LangChain connects the components.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu \
  langchain langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
import locale

locale.getpreferredencoding = lambda: "UTF-8"
print("Locale:", locale.getpreferredencoding())

## 3. Enter a GitHub personal access token

Create a fine-grained token with read access to public repositories. `getpass` hides the value and keeps it out of the notebook source.

In [ ]:
from getpass import getpass

ACCESS_TOKEN = getpass("GitHub personal access token: ")
print("Token received:", bool(ACCESS_TOKEN))

## 4. Load GitHub issues

The loader requests open and closed issues from `huggingface/peft`. Pull requests are excluded because GitHub's API otherwise treats them as issues.

In [ ]:
from langchain_community.document_loaders import GitHubIssuesLoader

loader = GitHubIssuesLoader(
    repo="huggingface/peft",
    access_token=ACCESS_TOKEN,
    include_prs=False,
    state="all",
)

docs = loader.load()
print("Issues loaded:", len(docs))

In [ ]:
first_doc = docs[0]
print("Metadata keys:", first_doc.metadata.keys())
print("Metadata:", first_doc.metadata)
print("Content preview:\n", first_doc.page_content[:1000])

## 5. Split issues into overlapping chunks

Embedding models have input limits. Chunking also makes retrieval more precise. A 30-character overlap preserves some context across adjacent chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=30,
)

chunked_docs = splitter.split_documents(docs)
print("Original issues:", len(docs))
print("Chunks created:", len(chunked_docs))
print("First chunk length:", len(chunked_docs[0].page_content))
print("First chunk preview:\n", chunked_docs[0].page_content[:600])

## 6. Embed chunks and create the FAISS vector store

`BAAI/bge-base-en-v1.5` maps text to dense vectors. FAISS stores those vectors and performs nearest-neighbor similarity search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embedding_model.embed_query("How do PEFT adapters work?")
print("Embedding dimensions:", len(sample_vector))

db = FAISS.from_documents(chunked_docs, embedding_model)
print("FAISS vector store created.")

## 7. Create and inspect the retriever

Similarity search embeds the question and returns the four nearest document chunks.

In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

question = "How do you combine multiple adapters?"
retrieved_docs = retriever.invoke(question)

print("Retrieved chunks:", len(retrieved_docs))
for index, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- RESULT {index} ---")
    print("Source:", doc.metadata.get("url", doc.metadata.get("source", "unknown")))
    print(doc.page_content[:700])

## 8. Load Zephyr-7B in 4-bit precision

Quantization reduces memory use enough for a Colab T4. The original cookbook uses `bfloat16`; this notebook uses `float16`, which is compatible with T4 GPUs.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model device:", model.device)

## 9. Wrap the model as a LangChain LLM

The Transformers generation pipeline defines decoding behavior. `HuggingFacePipeline` adapts it to LangChain's Runnable interface.

In [ ]:
from transformers import pipeline
from langchain_huggingface.llms import HuggingFacePipeline

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=300,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("LangChain LLM is ready.")

## 10. Create the prompt and generation chain

Zephyr expects its chat control tokens. The prompt instructs the model to use retrieved context and admit when the context does not contain an answer.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt_template = """<|system|>
Answer the question using the supplied context. If the context does not contain
the answer, say that you do not know. Do not invent repository-specific facts.

Context:
{context}
</s>
<|user|>
{question}
</s>
<|assistant|>
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()
print(prompt.format(context="EXAMPLE CONTEXT", question="EXAMPLE QUESTION"))

## 11. Build the RAG chain

The retriever returns `Document` objects. `format_docs` converts them into a readable context string before the prompt is rendered.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(documents):
    return "\n\n".join(
        f"Source {index}:\n{doc.page_content}"
        for index, doc in enumerate(documents, start=1)
    )

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | llm_chain
)

print("RAG chain is ready.")

## 12. Compare the model without and with retrieval

The same LLM is used in both calls. Only the retrieved context changes.

In [ ]:
baseline_answer = llm_chain.invoke({
    "context": "",
    "question": question,
})

print("WITHOUT RETRIEVAL\n")
print(baseline_answer)

In [ ]:
rag_answer = rag_chain.invoke(question)

print("WITH RETRIEVAL\n")
print(rag_answer)

## 13. Return both the answer and source documents

A production RAG interface should retain its evidence so users can inspect where the answer came from.

In [ ]:
def answer_with_sources(user_question):
    source_docs = retriever.invoke(user_question)
    context = format_docs(source_docs)
    answer = llm_chain.invoke({"context": context, "question": user_question})
    return answer, source_docs

answer, sources = answer_with_sources(question)
print("ANSWER\n", answer)

print("\nSOURCES")
for index, doc in enumerate(sources, start=1):
    print(index, doc.metadata.get("url", doc.metadata.get("source", "unknown")))

## 14. Student experiments

Try questions that are specific to PEFT issues. For each question, inspect the retrieved chunks before trusting the generated answer.

In [ ]:
student_questions = [
    "Can multiple LoRA adapters be loaded for inference?",
    "What problems have users reported when merging adapters?",
]

for student_question in student_questions:
    answer, sources = answer_with_sources(student_question)
    print("\nQUESTION:", student_question)
    print("ANSWER:", answer)
    print("SOURCE COUNT:", len(sources))

## Key takeaway

RAG does not retrain Zephyr. It retrieves semantically related GitHub issue chunks and places them in the prompt. Answer quality therefore depends on loading, chunking, embeddings, retrieval, prompt design, generation, and evidence inspection.